In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import IsolationForest
import lightgbm as lgb

# --- 하이퍼파라미터 및 설정 설정 ---
SEED = 42
N_SPLITS = 5
SEEDS_MULTI = [42, 2024, 7, 100, 2025]
DECAY_HALF_LIFE = 13
LTV_TWEEDIE_POWER = 1.3952884526971863

# 이탈(Churn) 예측 모델 하이퍼파라미터
CHURN_PARAMS = {
    'learning_rate': 0.059990687581278165,
    'num_leaves': 8,
    'max_depth': 5,
    'min_child_samples': 93,
    'subsample': 0.9229353236536758,
    'colsample_bytree': 0.8895806575636213,
    'reg_alpha': 6.320173779921072e-07,
    'reg_lambda': 0.00026780509121074965,
    'n_estimators': 2000,
}

# [1/5] 원본 데이터 로딩
print("\n[1/5] 데이터 로딩 중...")
train_info = pd.read_csv('train_customer_info.csv')
train_finance = pd.read_csv('train_finance_profile.csv')
train_trans = pd.read_csv('train_transaction_history.csv')
train_targets = pd.read_csv('train_targets.csv')
test_info = pd.read_csv('test_customer_info.csv')
test_finance = pd.read_csv('test_finance_profile.csv')
test_trans = pd.read_csv('test_transaction_history.csv')

# [2/5] 피처 엔지니어링 (Feature Engineering)
print("[2/5] 피처 엔지니어링 수행 중...")
train_trans['trans_date'] = pd.to_datetime(train_trans['trans_date'])
test_trans['trans_date'] = pd.to_datetime(test_trans['trans_date'])

# 전체 데이터 기준 최신 거래일 설정 및 병합
global_max_date = max(train_trans['trans_date'].max(), test_trans['trans_date'].max())
all_trans = pd.concat([train_trans, test_trans], axis=0).reset_index(drop=True)

# 시간 경과에 따른 가중치 산출 (시간 감쇠 효과 반영)
all_trans['days_ago'] = (global_max_date - all_trans['trans_date']).dt.days
all_trans['decay_weight'] = np.exp(-all_trans['days_ago'] / DECAY_HALF_LIFE)
all_trans['decayed_amount'] = all_trans['trans_amount'] * all_trans['decay_weight']

# 기본 거래 데이터 집계 (Aggregation)
agg = all_trans.groupby('customer_id').agg({
    'trans_amount': ['sum', 'mean', 'max', 'std', 'median'],
    'decayed_amount': ['sum', 'mean'],
    'trans_id': ['count'],
    'trans_date': ['max', 'min'],
    'is_installment': ['sum', 'mean'],
})
agg.columns = ['_'.join(c).strip() for c in agg.columns.values]
agg = agg.rename(columns={'trans_id_count': 'transaction_count'}).reset_index()

# 최근성(Recency) 및 구매 주기(Purchase Cycle) 산출
agg['recency_days'] = (global_max_date - agg['trans_date_max']).dt.days
agg['purchase_cycle_days'] = ((agg['trans_date_max'] - agg['trans_date_min']).dt.days / (agg['transaction_count'] - 1).clip(lower=1))

# --- 파생 변수 생성: 이탈 저항력 백분위 점수 (Churn Resistance Score) ---
# R(최근성), F(빈도), M(금액) 요소를 전체 고객 기준 백분위(0~1)로 변환하여 아웃라이어 영향 최소화
r_rank = agg['recency_days'].rank(pct=True)       # 높을수록 이탈 위험 상승 (부정 지표)
f_rank = agg['transaction_count'].rank(pct=True)  # 높을수록 충성도 상승 (긍정 지표)
m_rank = agg['trans_amount_sum'].rank(pct=True)   # 높을수록 충성도 상승 (긍정 지표)

# 통합 이탈 저항력 점수 정의 (R은 역산 적용)
agg['churn_resistance_score'] = (f_rank + m_rank + (1.0 - r_rank)) / 3.0
# ----------------------------------------------------------------------

agg['silence_risk_score'] = agg['recency_days'] / (agg['purchase_cycle_days'] + 1e-5)
agg['spend_cv'] = agg['trans_amount_std'].fillna(0) / (agg['trans_amount_mean'] + 1e-5)
agg.drop(columns=['trans_date_max', 'trans_date_min'], inplace=True)

# 단기(14일) 및 장기(90일) 거래 트렌드 변수 생성
for w, name in [(14, 'recent'), (90, 'long_term')]:
    sub = all_trans[all_trans['days_ago'] <= w].groupby('customer_id')['trans_amount'].sum().reset_index()
    sub.rename(columns={'trans_amount': f'amount_{name}'}, inplace=True)
    agg = agg.merge(sub, on='customer_id', how='left').fillna({f'amount_{name}': 0})

agg['recent_momentum_ratio'] = agg['amount_recent'] / (agg['trans_amount_mean'] * 14 + 1e-5)
agg['long_term_loyalty_ratio'] = agg['amount_long_term'] / (agg['trans_amount_mean'] * 90 + 1e-5)

# 카테고리 텍스트 임베딩 차원 축소 (SVD)
all_trans['item_category'] = all_trans['item_category'].astype(str).fillna('Unknown').str.replace(' ', '_')
seq = all_trans.groupby('customer_id')['item_category'].apply(lambda x: ' '.join(x)).reset_index()
tfidf = TfidfVectorizer(max_features=100, ngram_range=(1, 2), token_pattern=r"(?u)\b\w+\b")
mat = tfidf.fit_transform(seq['item_category'])
svd = TruncatedSVD(n_components=10, random_state=42)
svd_mat = svd.fit_transform(mat)
svd_df = pd.DataFrame(svd_mat, columns=[f'svd_{i}' for i in range(10)])
svd_df['customer_id'] = seq['customer_id'].values
agg = agg.merge(svd_df, on='customer_id', how='left')

# 고객 정보 및 금융 프로필 병합 함수
def merge_all(info, fin, ag, tgt=None):
    m = info.merge(fin, on='customer_id', how='left').merge(ag, on='customer_id', how='left')
    m['join_date'] = pd.to_datetime(m['join_date'])
    m['customer_tenure_days'] = (global_max_date - m['join_date']).dt.days
    
    if 'income_group' in m.columns:
        m['income_group_num'] = m['income_group'].map({'G1':1, 'G2':2, 'G3':3, 'G4':4, 'G5':5}).fillna(0)
        m.drop(columns=['income_group'], inplace=True)
        
    for c in ['total_deposit_balance', 'total_loan_balance', 'card_cash_service_amt', 'card_loan_amt', 'fin_overdue_days', 'fin_asset_trend_score', 'credit_score', 'num_active_cards']:
        if c in m.columns: 
            m[c] = m[c].fillna(0)
            
    m['debt_to_asset_ratio'] = ((m['total_loan_balance'] + m['card_cash_service_amt'] + m['card_loan_amt']) / (m['total_deposit_balance'] + 1e-6))
    m['short_term_debt_ratio'] = ((m['card_cash_service_amt'] + m['card_loan_amt']) / (m['total_deposit_balance'] + 1e-6))
    
    for c in ['join_date', 'gender', 'region_code', 'marital_status', 'is_married']:
        if c in m.columns: 
            m.drop(columns=[c], inplace=True)
            
    if tgt is not None: 
        m = m.merge(tgt, on='customer_id', how='left')
    return m

train_df = merge_all(train_info, train_finance, agg[agg['customer_id'].isin(train_info['customer_id'])], train_targets)
test_df = merge_all(test_info, test_finance, agg[agg['customer_id'].isin(test_info['customer_id'])])

# 범주형 변수 인코딩 및 Isolation Forest 기반 이상치 점수 산출
for col in train_df.select_dtypes(exclude=['number']).columns:
    if col == 'customer_id': 
        continue
    le = LabelEncoder()
    le.fit(pd.concat([train_df[col], test_df[col]], axis=0).astype(str))
    train_df[col] = le.transform(train_df[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))

features = [c for c in train_df.columns if c not in ['customer_id', 'target_churn', 'target_ltv']]
iso = IsolationForest(n_estimators=100, contamination=0.03, random_state=42)
iso.fit(pd.concat([train_df[features], test_df[features]], axis=0).fillna(0))
train_df['anomaly_score'] = iso.decision_function(train_df[features].fillna(0))
test_df['anomaly_score'] = iso.decision_function(test_df[features].fillna(0))
features.append('anomaly_score')

# [3/5] Multi-seed 앙상블 모델 학습
print(f"\n[3/5] Multi-seed 학습 시작 ({len(SEEDS_MULTI)} seeds)...")
X, X_test = train_df[features].fillna(0), test_df[features].fillna(0)
y_churn, y_ltv = train_df['target_churn'], train_df['target_ltv']
all_test_churn, all_test_ltv, seed_aucs, seed_rmses = [], [], [], []

for si, seed in enumerate(SEEDS_MULTI):
    print(f"   --- Seed {seed} ({si+1}/{len(SEEDS_MULTI)}) ---")
    
    # Task 1: Churn 예측 (분류)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    oof_c, test_c = np.zeros(len(X)), np.zeros(len(X_test))
    params = CHURN_PARAMS.copy()
    params['random_state'] = seed

    for tr, va in skf.split(X, y_churn):
        m = lgb.LGBMClassifier(**params)
        m.fit(X.iloc[tr], y_churn.iloc[tr], eval_set=[(X.iloc[va], y_churn.iloc[va])], callbacks=[lgb.early_stopping(50, verbose=False)])
        oof_c[va] = m.predict_proba(X.iloc[va])[:, 1]
        test_c += m.predict_proba(X_test)[:, 1] / 5
    
    auc = roc_auc_score(y_churn, oof_c)
    seed_aucs.append(auc)
    all_test_churn.append(test_c)
    print(f"       Churn AUC: {auc:.5f}")

    # Task 2: LTV 예측 (회귀) - Churn 예측 결과를 메타 피처로 활용
    X_ltv, X_test_ltv = X.copy(), X_test.copy()
    X_ltv['churn_risk'], X_test_ltv['churn_risk'] = oof_c, test_c
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    oof_l, test_l = np.zeros(len(X)), np.zeros(len(X_test))
    
    for tr, va in kf.split(X_ltv):
        m = lgb.LGBMRegressor(n_estimators=1500, learning_rate=0.02, max_depth=6, objective='tweedie', tweedie_variance_power=LTV_TWEEDIE_POWER, random_state=seed, verbose=-1)
        m.fit(X_ltv.iloc[tr], y_ltv.iloc[tr], eval_set=[(X_ltv.iloc[va], y_ltv.iloc[va])], callbacks=[lgb.early_stopping(50, verbose=False)])
        oof_l[va] = m.predict(X_ltv.iloc[va])
        test_l += m.predict(X_test_ltv) / 5
    
    rmse = np.sqrt(mean_squared_error(y_ltv, np.clip(oof_l, 0, None)))
    seed_rmses.append(rmse)
    all_test_ltv.append(np.clip(test_l, 0, None))
    print(f"       LTV RMSE:  {rmse:,.2f}")

# [5/5] 검증 성능 요약 및 최종 결과 파일 저장
mean_auc = np.mean(seed_aucs)
mean_rmse = np.mean(seed_rmses)
print(f"\n[5/5] 최종 모델 성능 요약")
print("=" * 60)
print(f"   Churn AUC  : {mean_auc:.5f}  (std: {np.std(seed_aucs):.5f})")
print(f"   LTV  RMSE  : {mean_rmse:,.2f}")
print("=" * 60)
for i, seed in enumerate(SEEDS_MULTI):
    print(f"     Seed {seed:>5}: AUC {seed_aucs[i]:.5f}  RMSE {seed_rmses[i]:>12,.2f}")

# 예측 결과 파일 내보내기
submission = pd.DataFrame({
    'customer_id': test_df['customer_id'],
    'target_churn': np.mean(all_test_churn, axis=0),
    'target_ltv': np.mean(all_test_ltv, axis=0)
})
submission.to_csv("제타바이트_submission_5.csv", index=False)
print("\n 제타바이트_submission_5.csv 생성 완료!")